# 04 — Model Comparison

Final comparison of all models with selection reasoning.

Results are loaded from the verified model comparison file.

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load comparison results
with open('../models/production/model_comparison_full.json') as f:
    comparison = json.load(f)

models_data = comparison['models']
print(f'Best model: {comparison["best_model"]}')
print(f'Data source: {comparison.get("data_source", "hopsworks")}')
print(f'Train rows: {comparison.get("train_rows", "N/A")}')
print(f'Val rows: {comparison.get("val_rows", "N/A")}')
print(f'Test rows: {comparison.get("test_rows", "N/A")}')
print(f'Features: {comparison.get("features", "N/A")}')

## 1. Overall Performance

In [ ]:
# Comparison table
table = []
for name, data in models_data.items():
    test = data.get('test_metrics', {})
    table.append({
        'Model': name,
        'Test MAE': f'{test.get("mae", 0):.2f}',
        'Test RMSE': f'{test.get("rmse", 0):.2f}',
        'Test R²': f'{test.get("r2", 0):.4f}',
        'Train Time': f'{data.get("train_time", 0):.1f}s',
    })
pd.DataFrame(table)

In [ ]:
# MAE comparison bar chart
model_names = list(models_data.keys())
mae_values = [models_data[m]['test_metrics']['mae'] for m in model_names]
r2_values = [models_data[m]['test_metrics']['r2'] for m in model_names]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Test MAE (Lower is Better)', 'Test R² (Higher is Better)'])

colors = ['#2ecc71' if m == 'XGBoost' else '#3498db' for m in model_names]

fig.add_trace(go.Bar(x=model_names, y=mae_values, marker_color=colors, name='MAE'), row=1, col=1)
fig.add_trace(go.Bar(x=model_names, y=r2_values, marker_color=colors, name='R²'), row=1, col=2)

fig.update_layout(height=400, showlegend=False)
fig.show()

## 2. Per-Horizon Comparison

In [ ]:
horizons = ['24h', '48h', '72h']
horizon_data = []
for h in horizons:
    for name in model_names:
        m = models_data[name]['test_metrics']
        horizon_data.append({
            'Horizon': h,
            'Model': name,
            'MAE': m[f'mae_{h}'],
            'RMSE': m[f'rmse_{h}'],
            'R²': m[f'r2_{h}'],
        })

hdf = pd.DataFrame(horizon_data)
print(hdf.to_string(index=False))

In [ ]:
# Per-horizon MAE bar chart
fig = go.Figure()
for name in model_names:
    h_mae = [hdf[(hdf['Model'] == name) & (hdf['Horizon'] == h)]['MAE'].values[0] for h in horizons]
    fig.add_trace(go.Bar(name=name, x=horizons, y=h_mae))
fig.update_layout(barmode='group', title='MAE by Horizon (Test Set)', yaxis_title='MAE', height=400)
fig.show()

## 3. Selection Reasoning

In [ ]:
# ── Selection Reasoning ────────────────────────────────────────
print('=== MODEL SELECTION REASONING ===')
print(f'Production Model: {comparison["best_model"]}')
best_key = comparison['best_model']
best_test = models_data[best_key]['test_metrics']
print(f'Test MAE: {best_test["mae"]:.2f}')
print(f'Test R²: {best_test["r2"]:.4f}')
print(f'Test RMSE: {best_test["rmse"]:.2f}')
print()
# Per-horizon winner
horizons = ['24h', '48h', '72h']
for h in horizons:
    best_h = min(model_names, key=lambda m: models_data[m]['test_metrics'][f'mae_{h}'])
    best_mae = models_data[best_h]['test_metrics'][f'mae_{h}']
    best_r2 = models_data[best_h]['test_metrics'][f'r2_{h}']
    print(f'{h}: {best_h} (MAE={best_mae:.2f}, R²={best_r2:.4f})')
print()
print('Why the selected model:')
print('1. Best overall composite score (MAE + RMSE + R²)')
print('2. Wins or is competitive on ALL 3 horizons')
print('3. Fast training — suitable for daily retraining')
print('4. Handles non-linear AQI patterns effectively')
print()
print('All models comparison:')
for name in sorted(model_names, key=lambda m: models_data[m]['test_metrics']['mae']):
    m = models_data[name]['test_metrics']
    print(f'  {name}: MAE={m["mae"]:.2f}, RMSE={m["rmse"]:.2f}, R²={m["r2"]:.4f}')
""")